# Plant Disease Prediction - MobileNetV2 (Transfer Learning)

## Importing Dataset

Dataset Link: https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset

## Importing libraries

In [ ]:
import json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
SEED = 42
EPOCHS = 15
FINE_TUNE = False

## Data Preprocessing

Train folder is split 90/10 into training and validation. The valid folder is kept aside as the test set
and is only used at the end.

### Training and Validation Image preprocessing

In [ ]:
training_set, validation_set = tf.keras.utils.image_dataset_from_directory(
    'train',
    labels="inferred",
    label_mode="categorical",
    color_mode="rgb",
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    shuffle=True,
    seed=SEED,
    validation_split=0.1,
    subset="both",
    interpolation="bilinear"
)

### Test Image Preprocessing

In [ ]:
test_set = tf.keras.utils.image_dataset_from_directory(
    'valid',
    labels="inferred",
    label_mode="categorical",
    color_mode="rgb",
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    shuffle=False,
    interpolation="bilinear"
)
class_name = test_set.class_names
print(len(class_name))

## Building Model

### Data Augmentation

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15, fill_mode="reflect"),
    layers.RandomZoom(0.1, fill_mode="reflect"),
    layers.RandomContrast(0.1),
], name="data_augmentation")

### Pre-trained MobileNetV2 base

In [ ]:
base_model = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE + (3,),
                                               include_top=False,
                                               weights="imagenet")
base_model.trainable = False

### Classifier head

In [ ]:
inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = layers.Rescaling(1.0 / 127.5, offset=-1.0)(x)   # MobileNetV2 expects pixels in [-1, 1]
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(512, activation="relu")(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(38, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

### Compiling and Training Phase

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6)
]
training_history = model.fit(x=training_set, validation_data=validation_set,
                             epochs=EPOCHS, callbacks=callbacks)
history = {k: [float(v) for v in vals] for k, vals in training_history.history.items()}

### Fine-tuning (optional)

In [ ]:
if FINE_TUNE:
    base_model.trainable = True
    for layer in base_model.layers[:-30]:
        layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    ft_history = model.fit(x=training_set, validation_data=validation_set,
                           epochs=5, callbacks=callbacks[:1])
    for k, vals in ft_history.history.items():
        history[k].extend(float(v) for v in vals)

### Saving Model

In [ ]:
model.save('trained_plant_disease_model.keras')

In [ ]:
#Recording History in json
with open('training_hist_mobilenetv2.json', 'w') as f:
    json.dump(history, f)

## Accuracy Visualization

In [ ]:
epochs = range(1, len(history['accuracy']) + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(epochs, history['accuracy'], color='red', label='Training Accuracy')
ax[0].plot(epochs, history['val_accuracy'], color='blue', label='Validation Accuracy')
ax[0].set_xlabel('No. of Epochs'); ax[0].set_title('Accuracy'); ax[0].legend()
ax[1].plot(epochs, history['loss'], color='red', label='Training Loss')
ax[1].plot(epochs, history['val_loss'], color='blue', label='Validation Loss')
ax[1].set_xlabel('No. of Epochs'); ax[1].set_title('Loss'); ax[1].legend()
plt.show()

## Evaluating Model on Test Set

In [ ]:
test_loss, test_acc = model.evaluate(test_set)
print('Test accuracy:', test_acc)

## Some other metrics for model evaluation

In [ ]:
y_pred = model.predict(test_set)
predicted_categories = np.argmax(y_pred, axis=1)
Y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for x, y in test_set])

In [ ]:
# Precision Recall Fscore
print(classification_report(Y_true, predicted_categories, target_names=class_name))

### Confusion Matrix Visualization

In [ ]:
cm = confusion_matrix(Y_true, predicted_categories)
plt.figure(figsize=(40, 40))
sns.heatmap(cm, annot=True, annot_kws={"size": 10})
plt.xlabel('Predicted Class', fontsize=20)
plt.ylabel('Actual Class', fontsize=20)
plt.title('Plant Disease Prediction Confusion Matrix', fontsize=25)
plt.show()